# `collab-sol` risk-model demo

This notebook loads only a small numeric feature panel from the final dataset, reserves the latest year as the untouched test set, tunes XGBoost over expanding temporal folds, and creates continuous risk scores for the final test year. Each `asof_year` row already contains a January 1 snapshot and that same calendar year's incident label; no label shifting or cross-year join is needed.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/Team 2/code/collab-sol")
DATASET_PATH = Path("/content/drive/MyDrive/Team 2/data/processed/final_dataset.parquet")
RUN_DIR = Path("/content/drive/MyDrive/Team 2/code/runs/xgboost_demo")
RUN_DIR.mkdir(parents=True, exist_ok=True)

%cd {PROJECT_ROOT}
%pip install -q .

In [ ]:
import json

import pandas as pd
from sklearn.impute import SimpleImputer

from collab_sol.evaluate import generate_risk_report
from collab_sol.models import XGBoostModel
from collab_sol.splitting import make_expanding_temporal_split
from collab_sol.tuning import tune_model

## Load selected columns

Parquet column selection avoids loading the hundreds of unused source columns. This is a demonstration feature list, not the final feature-selection decision. `incident_count` and all outcome/severity fields must be excluded because they are derived from the Form 57 outcomes being predicted.

In [ ]:
FEATURE_COLUMNS = [
    "trainsperweek",
    "maximumtimetablespeed",
    "numberofmaintracks",
    "trafficlane",
    "highwayspeedlimit",
    "annualaveragedailytrafficcount",
    "estimatedpercenttrucks",
    "pr_total_mm_5y_mean",
    "wet_days_5y_mean",
    "freeze_thaw_days_5y_mean",
    "hot_days_5y_mean",
    "tmin_mean_c_5y_mean",
    "tmax_mean_c_5y_mean",
    "wind_mean_ms_5y_mean",
]

LEAKAGE_COLUMNS = {
    "incident",
    "incident_count",
    "total_killed",
    "total_injured",
    "severity_ordinal",
    "severity_target",
}
leaked_features = LEAKAGE_COLUMNS.intersection(FEATURE_COLUMNS)
if leaked_features:
    raise ValueError(f"Outcome leakage in FEATURE_COLUMNS: {sorted(leaked_features)}")

frame = pd.read_parquet(
    DATASET_PATH,
    columns=["crossingid", "asof_year", "incident", *FEATURE_COLUMNS],
)

display(
    frame.groupby("asof_year")["incident"]
    .agg(rows="size", crashes="sum", prevalence="mean")
)

## Define temporal folds and tune

The latest year is never passed to Optuna. The imputer is refit inside every training fold, which prevents validation leakage. Increase `N_TRIALS` for reportable model selection.

In [ ]:
X = frame[FEATURE_COLUMNS]
y = frame["incident"].astype(int)
split = make_expanding_temporal_split(frame, year_column="asof_year")

def make_preprocessor():
    return SimpleImputer(strategy="median", add_indicator=True)

print("Validation years:", [fold.validation_year for fold in split.folds])
print("Untouched test year:", split.holdout_year)

In [ ]:
N_TRIALS = 2  # Smoke test only; increase this for the real comparison.

study = tune_model(
    model_class=XGBoostModel,
    X=X,
    y=y,
    split=split,
    preprocessor_factory=make_preprocessor,
    n_trials=N_TRIALS,
    study_name="xgboost_risk",
)

study.trials_dataframe().to_csv(RUN_DIR / "optuna_trials.csv", index=False)
print("Mean validation AP:", study.best_value)
print("Best parameters:", study.best_params)

## Refit on all development years and evaluate the test year once

In [ ]:
final_preprocessor = make_preprocessor()
X_development = final_preprocessor.fit_transform(
    X.iloc[split.development_indices],
    y.iloc[split.development_indices],
)
X_test = final_preprocessor.transform(X.iloc[split.holdout_indices])

final_model = XGBoostModel(params=study.best_params)
final_model.fit(
    X_development,
    y.iloc[split.development_indices].to_numpy(),
)

metrics = generate_risk_report(
    final_model,
    X_test,
    y.iloc[split.holdout_indices].to_numpy(),
    output_dir=RUN_DIR,
    row_ids=frame.iloc[split.holdout_indices]["crossingid"].to_numpy(),
)

run_metadata = {
    "model": final_model.name,
    "holdout_year": split.holdout_year,
    "features": FEATURE_COLUMNS,
    "best_params": study.best_params,
}
(RUN_DIR / "run.json").write_text(json.dumps(run_metadata, indent=2))

display(pd.DataFrame([metrics.to_dict()]))
print("Report saved to:", RUN_DIR)